In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xml.etree.ElementTree as ET
from datasets import load_dataset
from collections import Counter
from tqdm.auto import tqdm
import re
import nltk

from nltk.tokenize import word_tokenize
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings('ignore')

# Download NLTK resources if needed
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
nltk.download('punkt_tab')

# Set up the plotting style
plt.style.use('ggplot')
sns.set(font_scale=1.2)
sns.set_style("whitegrid")

# Create a directory for saving figures
os.makedirs("eda_figures", exist_ok=True)

# Helper functions for text analysis
def count_tokens(text):
    """Count the number of tokens in a text."""
    return len(word_tokenize(text))

def count_words(text):
    """Count the number of words in a text."""
    return len(re.findall(r'\b\w+\b', text))

def count_math_symbols(text):
    """Count math symbols in text."""
    math_symbols = ['+', '-', '*', '/', '=', '<', '>', '≤', '≥', '×', '÷', '√', '^', '∫', '∑', '∏', 'π', '∞']
    return sum(text.count(symbol) for symbol in math_symbols)

def extract_numbers(text):
    """Extract all numbers from text."""
    return re.findall(r'\b\d+\.?\d*\b', text)

def analyze_text_data(data, text_column, title):
    """General analysis for text data."""
    print(f"\n{'='*80}\n{title} Analysis\n{'='*80}")
    
    # Handle either a list of strings or a pandas Series
    if isinstance(data, list):
        texts = data
    else:
        texts = data[text_column].tolist()
    
    # Text length analysis
    char_lengths = [len(text) for text in texts]
    word_counts = [count_words(text) for text in texts]
    token_counts = [count_tokens(text) for text in texts]
    
    # Calculate statistics
    print(f"Total number of examples: {len(texts)}")
    print(f"\nText Length Statistics (characters):")
    print(f"  Min: {min(char_lengths)}")
    print(f"  Max: {max(char_lengths)}")
    print(f"  Mean: {np.mean(char_lengths):.2f}")
    print(f"  Median: {np.median(char_lengths):.2f}")
    
    print(f"\nWord Count Statistics:")
    print(f"  Min: {min(word_counts)}")
    print(f"  Max: {max(word_counts)}")
    print(f"  Mean: {np.mean(word_counts):.2f}")
    print(f"  Median: {np.median(word_counts):.2f}")
    
    # Create distributions plot
    plt.figure(figsize=(16, 6))
    
    plt.subplot(1, 3, 1)
    sns.histplot(char_lengths, kde=True)
    plt.title(f'Character Length Distribution')
    plt.xlabel('Character Length')
    plt.ylabel('Frequency')
    plt.axvline(np.median(char_lengths), color='red', linestyle='--')
    
    plt.subplot(1, 3, 2)
    sns.histplot(word_counts, kde=True)
    plt.title(f'Word Count Distribution')
    plt.xlabel('Word Count')
    plt.ylabel('Frequency')
    plt.axvline(np.median(word_counts), color='red', linestyle='--')
    
    plt.subplot(1, 3, 3)
    sns.histplot(token_counts, kde=True)
    plt.title(f'Token Count Distribution')
    plt.xlabel('Token Count')
    plt.ylabel('Frequency')
    plt.axvline(np.median(token_counts), color='red', linestyle='--')
    
    plt.tight_layout()
    plt.savefig(f"eda_figures/{title.lower().replace(' ', '_')}_length_distributions.png")
    plt.close()
    
    # Math symbols analysis
    math_symbol_counts = [count_math_symbols(text) for text in texts]
    
    print(f"\nMath Symbol Statistics:")
    print(f"  Min: {min(math_symbol_counts)}")
    print(f"  Max: {max(math_symbol_counts)}")
    print(f"  Mean: {np.mean(math_symbol_counts):.2f}")
    print(f"  Median: {np.median(math_symbol_counts):.2f}")
    
    # Plot math symbol distribution
    plt.figure(figsize=(10, 6))
    sns.histplot(math_symbol_counts, kde=True)
    plt.title(f'Math Symbol Count Distribution')
    plt.xlabel('Number of Math Symbols')
    plt.ylabel('Frequency')
    plt.axvline(np.median(math_symbol_counts), color='red', linestyle='--')
    plt.savefig(f"eda_figures/{title.lower().replace(' ', '_')}_math_symbols.png")
    plt.close()
    
    # Sample a random example
    if len(texts) > 0:
        sample_idx = np.random.randint(0, len(texts))
        print(f"\nSample text (index {sample_idx}):")
        print(f"{texts[sample_idx][:1000]}{'...' if len(texts[sample_idx]) > 1000 else ''}")
    
    # Return the analysis results
    return {
        'count': len(texts),
        'char_length_stats': {
            'min': min(char_lengths),
            'max': max(char_lengths),
            'mean': np.mean(char_lengths),
            'median': np.median(char_lengths)
        },
        'word_count_stats': {
            'min': min(word_counts),
            'max': max(word_counts),
            'mean': np.mean(word_counts),
            'median': np.median(word_counts)
        },
        'token_count_stats': {
            'min': min(token_counts),
            'max': max(token_counts),
            'mean': np.mean(token_counts),
            'median': np.median(token_counts)
        },
        'math_symbol_stats': {
            'min': min(math_symbol_counts),
            'max': max(math_symbol_counts),
            'mean': np.mean(math_symbol_counts),
            'median': np.median(math_symbol_counts)
        }
    }

def analyze_qa_dataset(data, question_key, answer_key, title):
    """Analyze question-answer dataset."""
    print(f"\n{'='*80}\n{title} Analysis\n{'='*80}")
    
    # Data overview
    print(f"Total number of examples: {len(data)}")
    
    if isinstance(data, pd.DataFrame):
        questions = data[question_key].tolist()
        answers = data[answer_key].tolist()
    else:
        questions = [item[question_key] for item in data]
        answers = [item[answer_key] for item in data]
    
    # Analyze question and answer lengths
    q_char_lengths = [len(q) for q in questions]
    q_word_counts = [count_words(q) for q in questions]
    
    a_char_lengths = [len(a) for a in answers]
    a_word_counts = [count_words(a) for a in answers]
    
    # Math symbol counts
    q_math_symbols = [count_math_symbols(q) for q in questions]
    a_math_symbols = [count_math_symbols(a) for a in answers]
    
    # Print statistics
    print("\nQuestion Statistics:")
    print(f"  Character Length - Min: {min(q_char_lengths)}, Max: {max(q_char_lengths)}, Mean: {np.mean(q_char_lengths):.2f}, Median: {np.median(q_char_lengths):.2f}")
    print(f"  Word Count - Min: {min(q_word_counts)}, Max: {max(q_word_counts)}, Mean: {np.mean(q_word_counts):.2f}, Median: {np.median(q_word_counts):.2f}")
    print(f"  Math Symbols - Min: {min(q_math_symbols)}, Max: {max(q_math_symbols)}, Mean: {np.mean(q_math_symbols):.2f}, Median: {np.median(q_math_symbols):.2f}")
    
    print("\nAnswer Statistics:")
    print(f"  Character Length - Min: {min(a_char_lengths)}, Max: {max(a_char_lengths)}, Mean: {np.mean(a_char_lengths):.2f}, Median: {np.median(a_char_lengths):.2f}")
    print(f"  Word Count - Min: {min(a_word_counts)}, Max: {max(a_word_counts)}, Mean: {np.mean(a_word_counts):.2f}, Median: {np.median(a_word_counts):.2f}")
    print(f"  Math Symbols - Min: {min(a_math_symbols)}, Max: {max(a_math_symbols)}, Mean: {np.mean(a_math_symbols):.2f}, Median: {np.median(a_math_symbols):.2f}")
    
    # Plot distributions
    plt.figure(figsize=(16, 12))
    
    # Question length distribution
    plt.subplot(2, 2, 1)
    sns.histplot(q_char_lengths, kde=True)
    plt.title('Question Character Length')
    plt.xlabel('Character Length')
    plt.ylabel('Frequency')
    
    plt.subplot(2, 2, 2)
    sns.histplot(q_word_counts, kde=True)
    plt.title('Question Word Count')
    plt.xlabel('Word Count')
    plt.ylabel('Frequency')
    
    # Answer length distribution
    plt.subplot(2, 2, 3)
    sns.histplot(a_char_lengths, kde=True)
    plt.title('Answer Character Length')
    plt.xlabel('Character Length')
    plt.ylabel('Frequency')
    
    plt.subplot(2, 2, 4)
    sns.histplot(a_word_counts, kde=True)
    plt.title('Answer Word Count')
    plt.xlabel('Word Count')
    plt.ylabel('Frequency')
    
    plt.tight_layout()
    plt.savefig(f"eda_figures/{title.lower().replace(' ', '_')}_qa_distributions.png")
    plt.close()
    
    # Math symbols comparison
    plt.figure(figsize=(12, 6))
    data = pd.DataFrame({
        'Questions': q_math_symbols,
        'Answers': a_math_symbols
    })
    
    sns.boxplot(data=pd.melt(data), x='variable', y='value')
    plt.title('Math Symbol Distribution: Questions vs Answers')
    plt.xlabel('')
    plt.ylabel('Number of Math Symbols')
    plt.savefig(f"eda_figures/{title.lower().replace(' ', '_')}_math_symbols_comparison.png")
    plt.close()
    
    # Sample a random example
    if len(questions) > 0:
        sample_idx = np.random.randint(0, len(questions))
        print(f"\nSample Q&A pair (index {sample_idx}):")
        print(f"Question: {questions[sample_idx]}")
        print(f"Answer: {answers[sample_idx]}")
    
    # Correlation between question and answer lengths
    plt.figure(figsize=(10, 6))
    plt.scatter(q_word_counts, a_word_counts, alpha=0.5)
    plt.title('Question vs Answer Length')
    plt.xlabel('Question Word Count')
    plt.ylabel('Answer Word Count')
    
    # Add regression line
    z = np.polyfit(q_word_counts, a_word_counts, 1)
    p = np.poly1d(z)
    plt.plot(sorted(q_word_counts), p(sorted(q_word_counts)), "r--", alpha=0.8)
    
    plt.savefig(f"eda_figures/{title.lower().replace(' ', '_')}_qa_correlation.png")
    plt.close()
    
    # Return analysis results
    return {
        'count': len(questions),
        'question_stats': {
            'char_length': {
                'min': min(q_char_lengths),
                'max': max(q_char_lengths),
                'mean': np.mean(q_char_lengths),
                'median': np.median(q_char_lengths)
            },
            'word_count': {
                'min': min(q_word_counts),
                'max': max(q_word_counts),
                'mean': np.mean(q_word_counts),
                'median': np.median(q_word_counts)
            },
            'math_symbols': {
                'min': min(q_math_symbols),
                'max': max(q_math_symbols),
                'mean': np.mean(q_math_symbols),
                'median': np.median(q_math_symbols)
            }
        },
        'answer_stats': {
            'char_length': {
                'min': min(a_char_lengths),
                'max': max(a_char_lengths),
                'mean': np.mean(a_char_lengths),
                'median': np.median(a_char_lengths)
            },
            'word_count': {
                'min': min(a_word_counts),
                'max': max(a_word_counts),
                'mean': np.mean(a_word_counts),
                'median': np.median(a_word_counts)
            },
            'math_symbols': {
                'min': min(a_math_symbols),
                'max': max(a_math_symbols),
                'mean': np.mean(a_math_symbols),
                'median': np.median(a_math_symbols)
            }
        }
    }

def extract_numerical_answers(answers):
    """Extract numerical answers from answer texts."""
    numbers = []
    for answer in answers:
        # Try to find a number at the end of the answer
        match = re.search(r'(\d+\.?\d*)$', answer)
        if match:
            numbers.append(float(match.group(1)))
        else:
            # Try to find any number in the answer
            matches = re.findall(r'\b(\d+\.?\d*)\b', answer)
            if matches:
                numbers.append(float(matches[-1]))  # Take the last number
    return numbers

def plot_answer_distribution(answers, title):
    """Plot distribution of numerical answers."""
    numbers = extract_numerical_answers(answers)
    
    if len(numbers) > 0:
        plt.figure(figsize=(12, 6))
        sns.histplot(numbers, kde=True)
        plt.title(f'{title} - Numerical Answer Distribution')
        plt.xlabel('Answer Value')
        plt.ylabel('Frequency')
        
        # Use log scale if range is large
        if max(numbers) / (min(numbers) + 1e-10) > 100:
            plt.xscale('log')
            plt.title(f'{title} - Numerical Answer Distribution (Log Scale)')
        
        plt.savefig(f"eda_figures/{title.lower().replace(' ', '_')}_answer_distribution.png")
        plt.close()
        
        print(f"\nNumerical Answer Statistics:")
        print(f"  Count: {len(numbers)} (out of {len(answers)} answers)")
        if len(numbers) > 0:
            print(f"  Min: {min(numbers)}")
            print(f"  Max: {max(numbers)}")
            print(f"  Mean: {np.mean(numbers):.2f}")
            print(f"  Median: {np.median(numbers):.2f}")
        
        return {
            'count': len(numbers),
            'percentage': len(numbers) / len(answers) * 100,
            'min': min(numbers) if numbers else None,
            'max': max(numbers) if numbers else None,
            'mean': np.mean(numbers) if numbers else None,
            'median': np.median(numbers) if numbers else None
        }
    else:
        print(f"\nNo numerical answers found in the dataset.")
        return {
            'count': 0,
            'percentage': 0
        }

# Function to extract math expressions
def extract_math_expressions(text):
    """Extract potential math expressions from text."""
    # This is a simplified pattern for basic math expressions
    pattern = r'\b\d+\s*[\+\-\*\/\=]\s*\d+'
    return re.findall(pattern, text)

# Function to analyze datasets for their mathematical content
def analyze_math_content(texts, title):
    """Analyze mathematical content in texts."""
    print(f"\n{'='*80}\n{title} Mathematical Content Analysis\n{'='*80}")
    
    # Count texts containing different math elements
    has_numbers = 0
    has_math_symbols = 0
    has_expressions = 0
    
    number_counts = []
    math_symbol_counts = []
    expression_counts = []
    
    for text in texts:
        # Count numbers
        numbers = re.findall(r'\b\d+\.?\d*\b', text)
        number_count = len(numbers)
        number_counts.append(number_count)
        if number_count > 0:
            has_numbers += 1
        
        # Count math symbols
        symbols = count_math_symbols(text)
        math_symbol_counts.append(symbols)
        if symbols > 0:
            has_math_symbols += 1
        
        # Count expressions
        expressions = extract_math_expressions(text)
        expression_counts.append(len(expressions))
        if len(expressions) > 0:
            has_expressions += 1
    
    total = len(texts)
    print(f"Total texts: {total}")
    print(f"Texts with numbers: {has_numbers} ({has_numbers/total*100:.2f}%)")
    print(f"Texts with math symbols: {has_math_symbols} ({has_math_symbols/total*100:.2f}%)")
    print(f"Texts with math expressions: {has_expressions} ({has_expressions/total*100:.2f}%)")
    
    print(f"\nNumber count stats: Min: {min(number_counts)}, Max: {max(number_counts)}, Mean: {np.mean(number_counts):.2f}")
    print(f"Math symbol count stats: Min: {min(math_symbol_counts)}, Max: {max(math_symbol_counts)}, Mean: {np.mean(math_symbol_counts):.2f}")
    print(f"Math expression count stats: Min: {min(expression_counts)}, Max: {max(expression_counts)}, Mean: {np.mean(expression_counts):.2f}")
    
    # Create combined plot
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    sns.histplot(number_counts, kde=False, discrete=True)
    plt.title('Number Count Distribution')
    plt.xlabel('Number of Numbers')
    plt.ylabel('Frequency')
    
    plt.subplot(1, 3, 2)
    sns.histplot(math_symbol_counts, kde=False, discrete=True)
    plt.title('Math Symbol Count Distribution')
    plt.xlabel('Number of Math Symbols')
    plt.ylabel('Frequency')
    
    plt.subplot(1, 3, 3)
    sns.histplot(expression_counts, kde=False, discrete=True)
    plt.title('Math Expression Count Distribution')
    plt.xlabel('Number of Math Expressions')
    plt.ylabel('Frequency')
    
    plt.tight_layout()
    plt.savefig(f"eda_figures/{title.lower().replace(' ', '_')}_math_content.png")
    plt.close()
    
    return {
        'total': total,
        'with_numbers': {
            'count': has_numbers,
            'percentage': has_numbers/total*100,
            'stats': {
                'min': min(number_counts),
                'max': max(number_counts),
                'mean': np.mean(number_counts),
                'median': np.median(number_counts)
            }
        },
        'with_math_symbols': {
            'count': has_math_symbols,
            'percentage': has_math_symbols/total*100,
            'stats': {
                'min': min(math_symbol_counts),
                'max': max(math_symbol_counts),
                'mean': np.mean(math_symbol_counts),
                'median': np.median(math_symbol_counts)
            }
        },
        'with_expressions': {
            'count': has_expressions,
            'percentage': has_expressions/total*100,
            'stats': {
                'min': min(expression_counts),
                'max': max(expression_counts),
                'mean': np.mean(expression_counts),
                'median': np.median(expression_counts)
            }
        }
    }

print("Starting EDA on Mathematical Datasets...")

# 1. Open Web Math Dataset
print("\n\n" + "="*80)
print("OPEN WEB MATH DATASET ANALYSIS")
print("="*80)

try:
    # Try to load samples for quicker analysis first
    sample_size = 10000  # Adjust based on your needs
    
    try:
        # Try to load local data first
        print(f"Attempting to load local Open Web Math data...")
        # Check common paths from your repository
        possible_paths = [
            "data/pre-training/open-web-math",
            "./data/pre-training/open-web-math",
            "math-reasoning-in-language-models/data/pre-training/open-web-math",
            "../data/pre-training/open-web-math"
        ]
        
        loaded = False
        for path in possible_paths:
            if os.path.exists(path):
                print(f"Found local data at {path}")
                # Use your own loading logic here if needed
                # For now, fall back to HF loading
                loaded = False
                break
        
        if not loaded:
            raise FileNotFoundError("Local data not found or not loadable")
                
    except Exception as e:
        print(f"Could not load local data: {e}")
        print(f"Loading from HuggingFace (sample of {sample_size} examples)...")
        owm_dataset = load_dataset("open-web-math/open-web-math", split=f"train[:{sample_size}]")
        print(f"Successfully loaded {len(owm_dataset)} examples from HuggingFace")
    
    # Basic dataset info
    print("\nDataset Structure:")
    print(f"Number of examples: {len(owm_dataset)}")
    print(f"Features: {owm_dataset.features}")
    
    # Sample examples
    print("\nSample examples:")
    for i in range(min(3, len(owm_dataset))):
        print(f"\nExample {i+1}:")
        example = owm_dataset[i]
        for key, value in example.items():
            if isinstance(value, str):
                print(f"{key}: {value[:200]}..." if len(value) > 200 else f"{key}: {value}")
            else:
                print(f"{key}: {value}")
    
    # Analyze text content
    owm_text_analysis = analyze_text_data(owm_dataset["text"], "text", "Open Web Math")
    
    # Analyze mathematical content
    owm_math_analysis = analyze_math_content(owm_dataset["text"], "Open Web Math")
    
    # URL Analysis if available
    if "url" in owm_dataset.features:
        urls = owm_dataset["url"]
        domains = [url.split('/')[2] if len(url.split('/')) > 2 else url for url in urls]
        domain_counts = Counter(domains)
        
        print("\nTop 10 domains:")
        for domain, count in domain_counts.most_common(10):
            print(f"  {domain}: {count} ({count/len(domains)*100:.2f}%)")
        
        plt.figure(figsize=(12, 6))
        domain_df = pd.DataFrame(domain_counts.most_common(10), columns=['Domain', 'Count'])
        sns.barplot(x='Count', y='Domain', data=domain_df)
        plt.title('Top 10 Domains in Open Web Math')
        plt.xlabel('Count')
        plt.tight_layout()
        plt.savefig("eda_figures/open_web_math_domains.png")
        plt.close()
    
except Exception as e:
    print(f"Error analyzing Open Web Math dataset: {e}")



# 3. TIGER-Lab/MathInstruct Dataset
print("\n\n" + "="*80)
print("TIGER-Lab/MathInstruct DATASET ANALYSIS")
print("="*80)

try:
    # Load MathInstruct dataset
    mathinstruct_dataset = load_dataset("TIGER-Lab/MathInstruct", split="train")
    print(f"Successfully loaded {len(mathinstruct_dataset)} examples")
    
    # Basic dataset info
    print("\nDataset Structure:")
    print(f"Number of examples: {len(mathinstruct_dataset)}")
    print(f"Features: {mathinstruct_dataset.features}")
    
    # Sample examples
    print("\nSample examples:")
    for i in range(min(3, len(mathinstruct_dataset))):
        print(f"\nExample {i+1}:")
        example = mathinstruct_dataset[i]
        for key, value in example.items():
            if isinstance(value, str):
                print(f"{key}: {value[:200]}..." if len(value) > 200 else f"{key}: {value}")
            else:
                print(f"{key}: {value}")
    
    # Analyze instruction-output pairs
    mathinstruct_qa_analysis = analyze_qa_dataset(
        mathinstruct_dataset, 
        'instruction', 
        'output', 
        'MathInstruct'
    )
    
    # Analyze distribution of answer values
    mathinstruct_answer_distribution = plot_answer_distribution(
        mathinstruct_dataset['output'], 
        'MathInstruct'
    )
    
    # Source distribution if available
    if 'source' in mathinstruct_dataset.features:
        sources = mathinstruct_dataset['source']
        source_counts = Counter(sources)
        
        print("\nSource Distribution:")
        for source, count in source_counts.most_common():
            print(f"  {source}: {count} ({count/len(sources)*100:.2f}%)")
        
        # Plot source distribution (top 10)
        plt.figure(figsize=(14, 8))
        source_df = pd.DataFrame(source_counts.most_common(10), columns=['Source', 'Count'])
        sns.barplot(x='Count', y='Source', data=source_df)
        plt.title('Top 10 Sources in MathInstruct')
        plt.xlabel('Count')
        plt.tight_layout()
        plt.savefig("eda_figures/mathinstruct_sources.png")
        plt.close()
        
        # Analyze math content by source (for top 5 sources)
        top_sources = [source for source, _ in source_counts.most_common(5)]
        
        for source in top_sources:
            source_examples = [example for example in mathinstruct_dataset if example['source'] == source]
            
            if len(source_examples) > 0:
                print(f"\nAnalyzing source: {source} ({len(source_examples)} examples)")
                instructions = [example['instruction'] for example in source_examples]
                outputs = [example['output'] for example in source_examples]
                
                # Brief analysis of this source
                i_lengths = [len(text) for text in instructions]
                o_lengths = [len(text) for text in outputs]
                
                print(f"  Instruction length: Mean: {np.mean(i_lengths):.2f}, Median: {np.median(i_lengths):.2f}")
                print(f"  Output length: Mean: {np.mean(o_lengths):.2f}, Median: {np.median(o_lengths):.2f}")
    
except Exception as e:
    print(f"Error analyzing MathInstruct dataset: {e}")

# 4. ASDiv Dataset
print("\n\n" + "="*80)
print("ASDiv DATASET ANALYSIS")
print("="*80)

try:
    # Attempt to find the local ASDiv XML file
    asdiv_paths = [
        "/Users/jonathan/Library/Mobile Documents/com~apple~CloudDocs/Master/Master Thesis/math-reasoning-in-language-models/data/curriculum_learning/1_ASDiv/ASDiv.xml",
        "./data/curriculum_learning/1_ASDiv/ASDiv.xml",
        "math-reasoning-in-language-models/data/curriculum_learning/1_ASDiv/ASDiv.xml",
        "../data/curriculum_learning/1_ASDiv/ASDiv.xml"
    ]
    
    asdiv_file = None
    for path in asdiv_paths:
        if os.path.exists(path):
            asdiv_file = path
            break
    
    if asdiv_file is None:
        print("ASDiv.xml file not found. Provide the correct path to analyze this dataset.")
    else:
        print(f"Found ASDiv file at: {asdiv_file}")
        
        # Parse the XML file
        tree = ET.parse(asdiv_file)
        root = tree.getroot()
        
        # Extract problems
        problems = []
        for problem in root.findall(".//Problem"):
            body_elem = problem.find("Body")
            question_elem = problem.find("Question")
            answer_elem = problem.find("Answer")
            
            if question_elem is not None and answer_elem is not None:
                body = body_elem.text.strip() if body_elem is not None else ""
                question = question_elem.text.strip()
                answer = answer_elem.text.strip()
                
                problems.append({
                    "body": body,
                    "question": question,
                    "answer": answer,
                    "full_question": f"{body} {question}" if body else question
                })
        
        print(f"Successfully extracted {len(problems)} problems from ASDiv.xml")
        
        # Basic analysis
        print("\nBasic statistics:")
        print(f"  Number of problems: {len(problems)}")
        
        # Convert to DataFrame for easier analysis
        asdiv_df = pd.DataFrame(problems)
        
        # Analyze question-answer pairs
        asdiv_qa_analysis = analyze_qa_dataset(
            asdiv_df, 
            'full_question', 
            'answer', 
            'ASDiv'
        )
        
        # Analyze the distribution of answer values
        asdiv_answer_distribution = plot_answer_distribution(
            asdiv_df['answer'].tolist(), 
            'ASDiv'
        )
        
        # Check for problem categories if present in the XML structure
        categories = []
        for problem in root.findall(".//Problem"):
            cat_elem = problem.find("category")
            if cat_elem is not None and cat_elem.text:
                categories.append(cat_elem.text.strip())
            else:
                categories.append("Uncategorized")
        
        if len(categories) == len(problems):
            category_counts = Counter(categories)
            
            print("\nProblem Categories:")
            for category, count in category_counts.most_common():
                print(f"  {category}: {count} ({count/len(categories)*100:.2f}%)")
            
            # Plot category distribution
            plt.figure(figsize=(14, 8))
            category_df = pd.DataFrame(category_counts.most_common(10), columns=['Category', 'Count'])
            sns.barplot(x='Count', y='Category', data=category_df)
            plt.title('Top 10 Categories in ASDiv')
            plt.xlabel('Count')
            plt.tight_layout()
            plt.savefig("eda_figures/asdiv_categories.png")
            plt.close()
        
except Exception as e:
    print(f"Error analyzing ASDiv dataset: {e}")

# 5. ParaMAWPS Dataset
print("\n\n" + "="*80)
print("ParaMAWPS DATASET ANALYSIS")
print("="*80)

try:
    # Attempt to find the local ParaMAWPS JSON file
    paramawps_paths = [
        "/Users/jonathan/Library/Mobile Documents/com~apple~CloudDocs/Master/Master Thesis/math-reasoning-in-language-models/data/curriculum_learning/2_ParaMAWPS/ParaMAWPS_trainset.json",
        "./data/curriculum_learning/2_ParaMAWPS/ParaMAWPS_trainset.json",
        "math-reasoning-in-language-models/data/curriculum_learning/2_ParaMAWPS/ParaMAWPS_trainset.json",
        "../data/curriculum_learning/2_ParaMAWPS/ParaMAWPS_trainset.json"
    ]
    
    paramawps_file = None
    for path in paramawps_paths:
        if os.path.exists(path):
            paramawps_file = path
            break
    
    if paramawps_file is None:
        print("ParaMAWPS_trainset.json file not found. Provide the correct path to analyze this dataset.")
    else:
        print(f"Found ParaMAWPS file at: {paramawps_file}")
        
        # Load the JSON file
        with open(paramawps_file, 'r') as f:
            paramawps_data = json.load(f)
        
        print(f"Successfully loaded {len(paramawps_data)} problems from ParaMAWPS_trainset.json")
        
        # Extract questions and answers
        questions = []
        answers = []
        equations = []
        
        for item in paramawps_data:
            questions.append(item.get("original_text", ""))
            answers.append(str(item.get("ans", "")))
            
            # Extract equation if available
            if "equation" in item:
                equations.append(item["equation"])
        
        print("\nBasic statistics:")
        print(f"  Number of problems: {len(questions)}")
        print(f"  Number with equations: {len(equations)}")
        
        # Analyze question-answer pairs
        paramawps_qa_data = [{"question": q, "answer": a} for q, a in zip(questions, answers)]
        paramawps_qa_analysis = analyze_qa_dataset(
            paramawps_qa_data, 
            'question', 
            'answer', 
            'ParaMAWPS'
        )
        
        # Analyze the distribution of answer values
        paramawps_answer_distribution = plot_answer_distribution(
            answers, 
            'ParaMAWPS'
        )
        
        # If equations are available, analyze them
        if equations:
            eq_lengths = [len(eq) for eq in equations]
            eq_symbols = [count_math_symbols(eq) for eq in equations]
            
            print("\nEquation Statistics:")
            print(f"  Length - Min: {min(eq_lengths)}, Max: {max(eq_lengths)}, Mean: {np.mean(eq_lengths):.2f}")
            print(f"  Math Symbols - Min: {min(eq_symbols)}, Max: {max(eq_symbols)}, Mean: {np.mean(eq_symbols):.2f}")
            
            # Plot equation statistics
            plt.figure(figsize=(12, 6))
            
            plt.subplot(1, 2, 1)
            sns.histplot(eq_lengths, kde=True)
            plt.title('Equation Length Distribution')
            plt.xlabel('Character Length')
            plt.ylabel('Frequency')
            
            plt.subplot(1, 2, 2)
            sns.histplot(eq_symbols, kde=True)
            plt.title('Math Symbols in Equations')
            plt.xlabel('Number of Math Symbols')
            plt.ylabel('Frequency')
            
            plt.tight_layout()
            plt.savefig("eda_figures/paramawps_equation_stats.png")
            plt.close()
        
except Exception as e:
    print(f"Error analyzing ParaMAWPS dataset: {e}")

# 6. DMath Dataset
print("\n\n" + "="*80)
print("DMath DATASET ANALYSIS")
print("="*80)

try:
    # Attempt to find the local DMath JSON file
    dmath_paths = [
        "/Users/jonathan/Library/Mobile Documents/com~apple~CloudDocs/Master/Master Thesis/math-reasoning-in-language-models/data/curriculum_learning/4_Dmath/dmath_train.json",
        "./data/curriculum_learning/4_Dmath/dmath_train.json",
        "math-reasoning-in-language-models/data/curriculum_learning/4_Dmath/dmath_train.json",
        "../data/curriculum_learning/4_Dmath/dmath_train.json"
    ]
    
    dmath_file = None
    for path in dmath_paths:
        if os.path.exists(path):
            dmath_file = path
            break
    
    if dmath_file is None:
        print("dmath_train.json file not found. Provide the correct path to analyze this dataset.")
    else:
        print(f"Found DMath file at: {dmath_file}")
        
        # Load the JSON file
        with open(dmath_file, 'r') as f:
            dmath_data = json.load(f)
        
        print(f"Successfully loaded DMath data with {len(dmath_data)} problems")
        
        # Extract questions and answers
        questions = []
        answers = []
        
        for item_id, item_data in dmath_data.items():
            questions.append(item_data.get("question_en", ""))
            answers.append(item_data.get("answer_en", ""))
        
        print("\nBasic statistics:")
        print(f"  Number of problems: {len(questions)}")
        
        # Analyze question-answer pairs
        dmath_qa_data = [{"question": q, "answer": a} for q, a in zip(questions, answers)]
        dmath_qa_analysis = analyze_qa_dataset(
            dmath_qa_data, 
            'question', 
            'answer', 
            'DMath'
        )
        
        # Analyze the distribution of answer values
        dmath_answer_distribution = plot_answer_distribution(
            answers, 
            'DMath'
        )
        
        # Check for additional fields in the first item
        sample_item_id = list(dmath_data.keys())[0]
        sample_item = dmath_data[sample_item_id]
        
        print("\nSample item structure:")
        for key, value in sample_item.items():
            if isinstance(value, str):
                print(f"  {key}: {value[:200]}..." if len(value) > 200 else f"  {key}: {value}")
            else:
                print(f"  {key}: {type(value)}")
        
        # Check if there are solution steps or other interesting fields
        if "solution_en" in sample_item:
            solution_lengths = [len(dmath_data[item_id].get("solution_en", "")) for item_id in dmath_data]
            
            print("\nSolution Statistics:")
            print(f"  Length - Min: {min(solution_lengths)}, Max: {max(solution_lengths)}, Mean: {np.mean(solution_lengths):.2f}")
            
            # Plot solution length distribution
            plt.figure(figsize=(10, 6))
            sns.histplot(solution_lengths, kde=True)
            plt.title('Solution Length Distribution in DMath')
            plt.xlabel('Character Length')
            plt.ylabel('Frequency')
            plt.savefig("eda_figures/dmath_solution_lengths.png")
            plt.close()
        
except Exception as e:
    print(f"Error analyzing DMath dataset: {e}")

# Cross-Dataset Comparison
print("\n\n" + "="*80)
print("CROSS-DATASET COMPARISON")
print("="*80)

try:
    # Create a comparison DataFrame
    datasets = []
    total_examples = []
    avg_question_lengths = []
    avg_answer_lengths = []
    avg_math_symbols = []
    
    # Try to add each dataset if we analyzed it
    try:
        datasets.append("OpenWebMath")
        total_examples.append(owm_text_analysis['count'])
        avg_question_lengths.append(owm_text_analysis['char_length_stats']['mean'])
        avg_answer_lengths.append(None)  # Not a QA dataset
        avg_math_symbols.append(owm_math_analysis['with_math_symbols']['stats']['mean'])
    except:
        pass
    
    try:
        datasets.append("FineWeb")
        total_examples.append(fineweb_text_analysis['count'])
        avg_question_lengths.append(fineweb_text_analysis['char_length_stats']['mean'])
        avg_answer_lengths.append(None)  # Not a QA dataset
        avg_math_symbols.append(fineweb_math_analysis['with_math_symbols']['stats']['mean'])
    except:
        pass
    
    try:
        datasets.append("MathInstruct")
        total_examples.append(mathinstruct_qa_analysis['count'])
        avg_question_lengths.append(mathinstruct_qa_analysis['question_stats']['char_length']['mean'])
        avg_answer_lengths.append(mathinstruct_qa_analysis['answer_stats']['char_length']['mean'])
        avg_math_symbols.append(mathinstruct_qa_analysis['question_stats']['math_symbols']['mean'])
    except:
        pass
    
    try:
        datasets.append("ASDiv")
        total_examples.append(asdiv_qa_analysis['count'])
        avg_question_lengths.append(asdiv_qa_analysis['question_stats']['char_length']['mean'])
        avg_answer_lengths.append(asdiv_qa_analysis['answer_stats']['char_length']['mean'])
        avg_math_symbols.append(asdiv_qa_analysis['question_stats']['math_symbols']['mean'])
    except:
        pass
    
    try:
        datasets.append("ParaMAWPS")
        total_examples.append(paramawps_qa_analysis['count'])
        avg_question_lengths.append(paramawps_qa_analysis['question_stats']['char_length']['mean'])
        avg_answer_lengths.append(paramawps_qa_analysis['answer_stats']['char_length']['mean'])
        avg_math_symbols.append(paramawps_qa_analysis['question_stats']['math_symbols']['mean'])
    except:
        pass
    
    try:
        datasets.append("DMath")
        total_examples.append(dmath_qa_analysis['count'])
        avg_question_lengths.append(dmath_qa_analysis['question_stats']['char_length']['mean'])
        avg_answer_lengths.append(dmath_qa_analysis['answer_stats']['char_length']['mean'])
        avg_math_symbols.append(dmath_qa_analysis['question_stats']['math_symbols']['mean'])
    except:
        pass
    
    # Create the comparison DataFrame if we have data
    if datasets:
        comparison_df = pd.DataFrame({
            'Dataset': datasets,
            'Total Examples': total_examples,
            'Avg Q Length': avg_question_lengths,
            'Avg A Length': avg_answer_lengths,
            'Avg Math Symbols': avg_math_symbols
        })
        
        print("Dataset Comparison Summary:")
        print(comparison_df.to_string(index=False))
        
        # Create comparison visualizations
        plt.figure(figsize=(14, 10))
        
        # Example count comparison
        plt.subplot(2, 2, 1)
        sns.barplot(x='Dataset', y='Total Examples', data=comparison_df)
        plt.title('Dataset Size Comparison')
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        # Question length comparison
        plt.subplot(2, 2, 2)
        sns.barplot(x='Dataset', y='Avg Q Length', data=comparison_df)
        plt.title('Average Question/Text Length')
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        # Answer length comparison (only for QA datasets)
        qa_comparison = comparison_df.dropna(subset=['Avg A Length'])
        plt.subplot(2, 2, 3)
        if not qa_comparison.empty:
            sns.barplot(x='Dataset', y='Avg A Length', data=qa_comparison)
            plt.title('Average Answer Length (QA Datasets)')
            plt.xticks(rotation=45)
            plt.tight_layout()
        
        # Math symbols comparison
        plt.subplot(2, 2, 4)
        sns.barplot(x='Dataset', y='Avg Math Symbols', data=comparison_df)
        plt.title('Average Math Symbols per Example')
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        plt.savefig("eda_figures/dataset_comparison.png")
        plt.close()
        
except Exception as e:
    print(f"Error in cross-dataset comparison: {e}")

print("\nEDA analysis complete! Results and visualizations saved to 'eda_figures' directory.")

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/jonathan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Starting EDA on Mathematical Datasets...


OPEN WEB MATH DATASET ANALYSIS
Attempting to load local Open Web Math data...
Could not load local data: Local data not found or not loadable
Loading from HuggingFace (sample of 10000 examples)...
Successfully loaded 10000 examples from HuggingFace

Dataset Structure:
Number of examples: 10000
Features: {'url': Value(dtype='string', id=None), 'text': Value(dtype='string', id=None), 'date': Value(dtype='string', id=None), 'metadata': Value(dtype='string', id=None)}

Sample examples:

Example 1:
url: https://telescoper.wordpress.com/2010/11/23/bayes-and-hi-theorem/
text: Bayes and his Theorem

My earlier post on Bayesian probability seems to have generated quite a lot of readers, so this lunchtime I thought I’d add a little bit of background. The previous discussion s...
date: 2016-09-28 22:12:05
metadata: {"extraction_info": {"found_math": true, "script_math_tex": 0, "script_math_asciimath": 0, "math_annotations": 0, "math_alttext": 0, "mathml"

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from collections import Counter
import pandas as pd

# Define both local and cloud directories for saving figures
local_dir = "domain_plots"
cloud_dir = "/Users/jonathan/Library/Mobile Documents/com~apple~CloudDocs/Master/Master Thesis/math-reasoning-in-language-models/plots/plots_images"

# Create directories for saving figures
os.makedirs(local_dir, exist_ok=True)
os.makedirs(cloud_dir, exist_ok=True)

print("Starting Domain Analysis...")

# 1. Open Web Math Dataset - Domain Analysis
print("\nAnalyzing OpenWebMath dataset domains...")

try:
    # Load a sample of OpenWebMath for domain analysis
    sample_size = 10000  # Adjust based on your needs
    owm_dataset = load_dataset("open-web-math/open-web-math", split=f"train[:{sample_size}]")
    print(f"Successfully loaded {len(owm_dataset)} examples from OpenWebMath")
    
    # URL Analysis
    if "url" in owm_dataset.features:
        urls = owm_dataset["url"]
        domains = [url.split('/')[2] if len(url.split('/')) > 2 else url for url in urls]
        domain_counts = Counter(domains)
        
        print(f"\nFound {len(domain_counts)} unique domains")
        print("\nTop 15 domains in OpenWebMath:")
        for domain, count in domain_counts.most_common(15):
            print(f"  {domain}: {count} ({count/len(domains)*100:.2f}%)")
        
        # Create plot for OpenWebMath domains
        plt.figure(figsize=(12, 8))
        domain_df = pd.DataFrame(domain_counts.most_common(15), columns=['Domain', 'Count'])
        sns.barplot(x='Count', y='Domain', data=domain_df)
        plt.title('Top 15 Domains in Open Web Math Dataset')
        plt.xlabel('Number of Examples')
        plt.tight_layout()
        
        # Save to both locations
        local_path = os.path.join(local_dir, "openwebmath_domains.png")
        cloud_path = os.path.join(cloud_dir, "openwebmath_domains.png")
        
        plt.savefig(local_path)
        plt.savefig(cloud_path)
        plt.close()
        
        print(f"OpenWebMath domain plot saved to:")
        print(f"  - {local_path}")
        print(f"  - {cloud_path}")
    else:
        print("No URL feature found in the OpenWebMath dataset")
except Exception as e:
    print(f"Error analyzing OpenWebMath dataset: {e}")

# 2. TIGER-Lab/MathInstruct Dataset - Source Analysis
print("\nAnalyzing MathInstruct dataset sources...")

try:
    # Load MathInstruct dataset
    mathinstruct_dataset = load_dataset("TIGER-Lab/MathInstruct", split="train")
    print(f"Successfully loaded {len(mathinstruct_dataset)} examples from MathInstruct")
    
    # Source distribution analysis
    if 'source' in mathinstruct_dataset.features:
        sources = mathinstruct_dataset['source']
        source_counts = Counter(sources)
        
        print(f"\nFound {len(source_counts)} unique sources")
        print("\nTop 15 sources in MathInstruct:")
        for source, count in source_counts.most_common(15):
            print(f"  {source}: {count} ({count/len(sources)*100:.2f}%)")
        
        # Plot source distribution (top 15)
        plt.figure(figsize=(14, 10))
        source_df = pd.DataFrame(source_counts.most_common(15), columns=['Source', 'Count'])
        sns.barplot(x='Count', y='Source', data=source_df)
        plt.title('Top 15 Sources in MathInstruct Dataset')
        plt.xlabel('Number of Examples')
        plt.tight_layout()
        
        # Save to both locations
        local_path = os.path.join(local_dir, "mathinstruct_sources.png")
        cloud_path = os.path.join(cloud_dir, "mathinstruct_sources.png")
        
        plt.savefig(local_path)
        plt.savefig(cloud_path)
        plt.close()
        
        print(f"MathInstruct source plot saved to:")
        print(f"  - {local_path}")
        print(f"  - {cloud_path}")
    else:
        print("No source feature found in the MathInstruct dataset")
except Exception as e:
    print(f"Error analyzing MathInstruct dataset: {e}")

print("\nDomain analysis complete! Plots saved to both directories.")